# M6: statistics, baselines, ablations (PLAN §7 M6; D-097)

**Notebook settings:** Accelerator **None (CPU)** · Internet **on** · Secrets: **HF_TOKEN**. Run as **Save Version → Save & Run All**.

Per config, the notebook:
1. restores the M4/M5 aggregates (`results/<run>/`: `delta_b_cells.csv.gz`, `bench_scores.csv`);
2. rescores WGM under the Qwen topic map from the private per-sample labels (`rescore-wgm`, D-095 sensitivity);
3. computes the target-only / attribute-only RR decomposition from the cached embeddings (`decompose-deltab`, D-085), restored to ephemeral `/tmp` (B-039).

Then `analyze` builds every M6 table (T1 per setting × benchmark × method, RR − baseline paired AUC, threshold sweep, ablations, WGM map × pairing, seeds, h-vs-u sign agreement, NF4 if its run exists) into `results/m6/`. The leakage test (DC-18) checks the results tree, which is synced to the private store and packed as `/kaggle/working/results_m6.tar.gz` (aggregates only) for download.

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "claude/kaggle-bias-detection-plan-h9es1w"  # a branch, tag or FULL 40-char SHA
WORK = "/kaggle/working/rbbd"
CONFIGS = ["tier2_qwen2.5-0.5b", "tier2_llama3.2-1b", "tier1_llama3.1-8b", "tier1_mistral-7b", "tier1_gemma3-4b"]
EXTRA_RUNS = ["m6_nf4_llama3.1-8b"]  # restored if present (M6-T6 GPU probe), compared in nf4.csv
DECOMPOSE = True                      # M6-T4: restores embeddings/<slug> to ephemeral /tmp (≈ 10 GB), not /kaggle/working
EPHEMERAL = "/tmp/rbbd_m6_cache"      # lost at session end; nothing persistent is written or deleted there
import os, time
SESSION_T0 = time.time()

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
# B-032 / D-087: the Kaggle image runs Python 3.13 and vllm 0.10.1.1 has no 3.13 wheels.
# The pinned stack goes into a Python 3.12 venv on ephemeral disk. Every uv call names its
# target interpreter explicitly, and the image's own UV_* settings are cleared first: they
# made uv install into the system Python (B-032, second run). Putting the venv's bin/ first
# on PATH makes every `!python ...` line and the stage subprocesses use it. This kernel
# only imports rbbd's light modules (config, runner, manifests) from src/.
import os, shutil, subprocess

def sh(*cmd, tail=25):
    # Run a setup command; show the end of its output; stop the notebook if it fails.
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print(f"$ {' '.join(cmd[:4])} ... -> exit {r.returncode}")
    if tail:
        print("\n".join(out[-tail:]))
    if r.returncode:
        raise RuntimeError(f"setup step failed: {' '.join(cmd[:4])}")
    return r.stdout

uv_vars = sorted(k for k in os.environ if k.startswith("UV_"))
print("image UV_* settings cleared:", uv_vars)  # names only
for k in uv_vars:
    del os.environ[k]
EPH = next((p for p in ("/kaggle/tmp", "/kaggle/temp") if os.path.isdir(p)), "/tmp")
VENV = f"{EPH}/rbbd_py312"
PY = f"{VENV}/bin/python"
os.environ.update(UV_CACHE_DIR=f"{EPH}/uv_cache", UV_LINK_MODE="copy",
                  UV_PYTHON_INSTALL_DIR=f"{EPH}/uv_python")
sh("pip", "install", "-q", "uv==0.8.17")
UV = shutil.which("uv")
sh(UV, "python", "install", "3.12")
sh(UV, "venv", "--python-preference", "only-managed", "--python", "3.12", VENV)
sh(PY, "--version")
sh(UV, "pip", "install", "--python", PY, "-e", ".[train,bench,dev]")
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
assert shutil.which("python") == PY, shutil.which("python")
with open("/kaggle/working/pip_freeze_m6.txt", "w") as fh:
    fh.write(sh(UV, "pip", "freeze", "--python", PY, tail=0))
# Preflight (B-012, D-049): stop here, not inside a stage, if the pinned stack does not import.
print(sh(PY, "-c", "import sys, torch, torchvision, transformers, vllm; print('preflight ok', "
         "sys.version.split()[0], torch.__version__, torchvision.__version__, "
         "transformers.__version__, vllm.__version__)"))

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# Restore aggregates + private inputs, then rescore / decompose per config. Aggregates only.
import json, shutil, subprocess
from pathlib import Path
from rbbd import config as _config
ROOT = Path("/kaggle/working/artifacts")
RES = Path(WORK) / "results"            # the checkout's results/ (what git and the leakage test see)
SUMMARY = {}
# The decomposition reads embed/, sentences/ and embeddings/<slug>: restored into EPHEMERAL
# (RBBD_ARTIFACTS for those subprocesses only), so /kaggle/working keeps only aggregates (B-039).
EPH_ENV = {**os.environ, "RBBD_ARTIFACTS": EPHEMERAL}
if DECOMPOSE:
    for path in ["embed", "sentences"]:
        !RBBD_ARTIFACTS={EPHEMERAL} python -m rbbd.cli restore --path {path}
for RUN in EXTRA_RUNS:
    !python -m rbbd.cli restore --path results/{RUN}
    if (ROOT / "results" / RUN).exists():
        shutil.copytree(ROOT / "results" / RUN, RES / RUN, dirs_exist_ok=True)
for NAME in CONFIGS:
    cfg = _config.load(f"configs/{NAME}.yaml")
    SLUG, RUN = cfg["models"][0]["slug"], cfg["run_name"]
    out = SUMMARY.setdefault(NAME, {})
    print(f"===== {NAME} ({SLUG}) =====")
    !python -m rbbd.cli restore --path results/{RUN}
    shutil.copytree(ROOT / "results" / RUN, RES / RUN, dirs_exist_ok=True)
    out["inputs"] = sorted(p.name for p in (RES / RUN).iterdir())
    !python -m rbbd.cli restore --path scores/wgm/{SLUG}
    r = subprocess.run(["python", "-m", "rbbd.cli", "rescore-wgm", "--config", f"configs/{NAME}.yaml"],
                       capture_output=True, text=True)
    out["rescore_wgm"] = r.stdout.strip()[-300:] if r.returncode == 0 else f"FAILED: {r.stderr[-800:]}"
    if DECOMPOSE:
        !RBBD_ARTIFACTS={EPHEMERAL} python -m rbbd.cli restore --path embeddings/{SLUG}
        r = subprocess.run(["python", "-m", "rbbd.cli", "decompose-deltab", "--config", f"configs/{NAME}.yaml"],
                           capture_output=True, text=True, env=EPH_ENV)
        out["decompose"] = r.stdout.strip()[-300:] if r.returncode == 0 else f"FAILED: {r.stderr[-800:]}"
    print(json.dumps(out, indent=1))
    !df -h /kaggle/working {EPHEMERAL} | tail -2
failed = [n for n, o in SUMMARY.items() if any(str(v).startswith("FAILED") for v in o.values())]
if failed:  # never build the tables from a partial set of inputs
    raise RuntimeError(f"rescore/decompose failed for {failed}; see the output above")

In [ ]:
# Every M6 table, then the DC-10 re-run (must be a cache hit) and the DC-18 leakage test.
t0 = time.time()
!python -m rbbd.cli analyze --results {RES} --out {RES}/m6 > /kaggle/working/m6_summary.json
SUMMARY["analyze_seconds"] = round(time.time() - t0, 1)
r = subprocess.run(["python", "-m", "rbbd.cli", "analyze", "--results", str(RES), "--out", str(RES / "m6")],
                   capture_output=True, text=True)
SUMMARY["dc10_cache_hit"] = "cache hit: analysis" in (r.stdout + r.stderr)
r = subprocess.run(["python", "-m", "pytest", "-q", "-p", "no:warnings", "tests/test_leakage.py"],
                   capture_output=True, text=True)
SUMMARY["dc18_leakage"] = (r.stdout.strip().splitlines() or ["?"])[-1]
if "failed" in SUMMARY["dc18_leakage"]:
    raise RuntimeError("leakage test failed; results/ not synced: " + r.stdout[-1500:])
shutil.copytree(RES, ROOT / "results", dirs_exist_ok=True)
for path in ["results"]:
    !python -m rbbd.cli sync --path {path}
!tar -czf /kaggle/working/results_m6.tar.gz -C {WORK} results
print(json.dumps(SUMMARY, indent=1))

In [ ]:
# Compact tables to paste back (aggregates only).
import csv
def rows(name):
    p = RES / "m6" / f"{name}.csv"
    return list(csv.DictReader(open(p))) if p.exists() else []
def f(x, n=3):
    try: return round(float(x), n)
    except (TypeError, ValueError): return x
print("== T1 (RR + baselines), pooled and per spectrum")
for r in rows("t1"):
    if r["method"] in ("rr", "seat", "procrustes", "cka", "rr_target", "rr_attr"):
        print(r["setting"], r["bench"], r["method"], "n", r["n_obs"], "pos", r["n_pos"],
              "r", f(r["pearson_r"]), r["stars"], "AUC", f(r["roc_auc"]), [f(r["auc_lo"]), f(r["auc_hi"])], "MAE", f(r["mae_in"]))
print("== C3: AUC(RR) - AUC(baseline), pooled settings")
for r in rows("c3"):
    if r["setting"].endswith("pooled"):
        print(r["setting"], r["bench"], r["a"], "-", r["b"], f(r["diff"]), [f(r["lo"]), f(r["hi"])])
print("== sign agreement (h - u)")
for r in rows("sign"):
    print(r["spectrum"], {k: f(v) for k, v in r.items() if k.startswith(("db_rr", "score_", "agree_rr")) and not k.endswith(("_lo", "_hi"))})
print("== WGM map x pairing (RR, pooled)")
for r in rows("wgm_pairing"):
    if r["method"] == "rr" and r["setting"].endswith("pooled"):
        print(r["map"], r["pairing"], r["setting"], "r", f(r["pearson_r"]), "AUC", f(r["roc_auc"]))
print("== seeds"); [print(r) for r in rows("seeds") if r["method"] == "rr"]
print("== nf4"); [print(r) for r in rows("nf4")]
print("rows per table:", {k: len(rows(k)) for k in ("t1", "c3", "f3_sweep", "ablations", "wgm_pairing", "sign", "seeds", "nf4")})
!python -m rbbd.cli sync --path env/$RBBD_SESSION_ID
print("session hours:", round((time.time() - SESSION_T0) / 3600, 2))
!df -h /kaggle/working | tail -1